# Approach v4: LSTM and Seq-to-Seq Deep Learning Models

In this approach, we explore replacing LightGBM with PyTorch-based Deep Learning architectures:
1. **LSTM Regressor Models (Single-Horizon)**: Trained separately for each target horizon (5m, 15m, 30m, 60m).
2. **Seq-to-Seq (Encoder-Decoder) Regressor Model (Multi-Horizon)**: Trained to autoregressively forecast the future 60 minutes of temperature values. Predictions for 5m, 15m, 30m, and 60m are extracted directly from the predicted sequence.

Both models are trained under two feature configurations:
- **All Features**: The 132 features engineered in Approach v3.
- **Selected Features**: The features selected via SHAP TreeExplainer in the New Approach (for Seq-to-Seq, we use the union of selected features across horizons).

Finally, we run `compare_v4.py` to evaluate and compare all metrics with the tuned LightGBM models.

## 1. Imports and Preprocessing Check

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

# Ensure project root is the working directory
if os.path.basename(os.getcwd()) == 'approch_v4':
    os.chdir('..')

from approch_v4.utils import preprocess_and_feature_engineering, get_feature_lists

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

df = preprocess_and_feature_engineering()
all_features, selected_features = get_feature_lists(df)
print(f"Data Shape: {df.shape}")
print(f"All features count: {len(all_features)}")
print(f"Selected features count (Union): {len(selected_features['union'])}")

## 2. Train LSTM Regressor Models

We run the training script for LSTM models. 
- To save training time on CPU, `train_sample_step` is set to `5` for Selected Features and `10` for All Features.
- Training runs for 8 epochs per horizon with early stopping.

In [ ]:
from approch_v4.train_lstm import train_and_eval_lstm_all_horizons

# Train LSTM on Selected Features (Sample endpoint every 5 minutes)
summary_lstm_sel = train_and_eval_lstm_all_horizons(df, feature_mode="selected", train_sample_step=5, epochs=8)

# Train LSTM on All Features (Sample endpoint every 10 minutes to save CPU time)
summary_lstm_all = train_and_eval_lstm_all_horizons(df, feature_mode="all", train_sample_step=10, epochs=8)

## 3. Train Seq-to-Seq Models

We run the training script for Encoder-Decoder Seq-to-Seq models. 
- The Encoder processes history while the Decoder predicts steps $t+1$ to $t+60$.
- Training is optimized using teacher forcing.

In [ ]:
from approch_v4.train_seq2seq import train_and_eval_seq2seq_all

# Train Seq-to-Seq on Selected Features (Union) (Sample endpoints every 5 mins)
summary_seq_sel = train_and_eval_seq2seq_all(df, feature_mode="selected", train_sample_step=5, epochs=8)

# Train Seq-to-Seq on All Features (Sample endpoints every 10 mins)
summary_seq_all = train_and_eval_seq2seq_all(df, feature_mode="all", train_sample_step=10, epochs=8)

## 4. Compile and Run Comparison

We run `compare_v4.py` to compile the performance metrics of all models into a single table.

In [ ]:
from approch_v4.compare_v4 import run_comparison
run_comparison()

## 5. Visualize Predictions on the Test Set

Let's load the predictions and plot the actual vs predicted temperature trajectories for a sample section of the test split (e.g., a known alarm episode period).

In [ ]:
# Load test predictions
pred_lstm_sel = pd.read_parquet('outputs/lstm_v4/predictions_lstm_selected.parquet')
pred_seq_sel = pd.read_parquet('outputs/seq2seq_v4/predictions_seq2seq_selected.parquet')
pred_lgb_sel = pd.read_parquet('outputs/tuned_new/comparison_predictions.parquet')

# Select a segment where an alarm occurs (actual >= 21.0)
alarm_times = pred_lstm_sel[pred_lstm_sel['actual'] >= 21.0].index
if len(alarm_times) > 0:
    sample_start = alarm_times[0] - pd.Timedelta(hours=2)
    sample_end = alarm_times[0] + pd.Timedelta(hours=4)
    
    # Plot 15-minute horizon prediction comparison
    plt.figure(figsize=(14, 6))
    plt.plot(pred_lstm_sel.loc[sample_start:sample_end].index, 
             pred_lstm_sel.loc[sample_start:sample_end, 'actual'], 
             label='Actual Temperature', color='black', linewidth=1.5)
    
    # LightGBM
    if 'pred_tuned_15m' in pred_lgb_sel.columns:
        plt.plot(pred_lgb_sel.loc[sample_start:sample_end].index, 
                 pred_lgb_sel.loc[sample_start:sample_end, 'pred_tuned_15m'], 
                 label='LightGBM 15m Pred (Selected Features)', color='blue', linestyle='--')
                 
    # LSTM
    if 'pred_15m' in pred_lstm_sel.columns:
        plt.plot(pred_lstm_sel.loc[sample_start:sample_end].index, 
                 pred_lstm_sel.loc[sample_start:sample_end, 'pred_15m'], 
                 label='LSTM 15m Pred (Selected Features)', color='green', linestyle='-.')
                 
    # Seq-to-Seq
    if 'pred_15m' in pred_seq_sel.columns:
        plt.plot(pred_seq_sel.loc[sample_start:sample_end].index, 
                 pred_seq_sel.loc[sample_start:sample_end, 'pred_15m'], 
                 label='Seq2Seq 15m Pred (Selected Features)', color='red', linestyle=':')
                 
    plt.axhline(21.0, color='red', linestyle='-', label='Alarm Threshold (21.0 degC)', alpha=0.5)
    plt.title('Prediction Comparison - 15-Minute Horizon')
    plt.xlabel('TimeStamp')
    plt.ylabel('Temperature (degC)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    # Save plot
    plt.savefig('outputs/v4/predictions_comparison_15m.png', dpi=150)
    plt.show()
else:
    print("No alarm times found in the predictions to plot.")